# Stage 2 -- Classical and transformation baselines on S1 (part of E1, E7)
Runs RF-BR, LGBM-BR, CC-ens, RAkEL and ML-kNN on all three feature views under the S1
protocol, three seeds (one in quick mode). Thresholds tuned on validation only.
**Attach:** `scvd-lib`, `dive-smart-contracts`, `scvd-stage1-eda`.

In [10]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/datasets/smnihalahmed/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage2_baselines')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

2026-09-24 11:41:11,652 INFO stage stage2_baselines initialised at /kaggle/working/stage2_baselines
2026-09-24 11:41:11,653 INFO seeds=(0, 1, 2) quick=False


In [11]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

2026-09-24 11:41:14,806 INFO loaded DIVE: pre=(22330, 222) post=(22330, 177) labels=(22330, 9) tools=(26071, 49)
2026-09-24 11:41:14,808 INFO family code_metadata        6 features
2026-09-24 11:41:14,809 INFO family interface_source    46 features
2026-09-24 11:41:14,810 INFO family creation_opcodes   168 features
2026-09-24 11:41:14,810 INFO family tx_metadata         10 features
2026-09-24 11:41:14,811 INFO family temporal_position    8 features
2026-09-24 11:41:14,812 INFO family runtime_opcodes    158 features
2026-09-24 11:41:14,812 INFO view F_pre  220 features
2026-09-24 11:41:14,814 INFO view F_post 176 features
2026-09-24 11:41:14,815 INFO view F_all  396 features


In [12]:
splits, sdf = scvd.load_splits()
if splits is None:
    log.warning('stage1 splits not found -- recomputing')
    splits = scvd.build_splits(pre, lab, seed=0)

In [ ]:
rows = []
res_path = f'{root}/results/stage2_baselines_results.csv'
done = set()
if os.path.exists(res_path):                    # resume after a kernel restart
    done = {tuple(r) for r in pd.read_csv(res_path)[['model', 'view', 'seed']].values}
    log.info('resuming: %d runs already checkpointed', len(done))
for view in scvd.VIEWS:
    d = scvd.make_data(pre, post, lab, fam, view, *splits['S1'])
    for model in ['RF-BR', 'LGBM-BR', 'CC-ens', 'RAkEL', 'ML-kNN']:
        for seed in SEEDS:
            if (model, view, seed) in done:
                log.info('skip %-8s F_%-4s seed%d (checkpointed)', model, view, seed)
                continue
            t0 = time.time()
            proba = scvd.fit_classical(model, d['X_tr'], d['y_tr'], seed)  # train ONCE
            th = scvd.tune_thresholds(d['y_va'], proba(d['X_va']))
            pt = proba(d['X_te'])
            m = scvd.all_metrics(d['y_te'], (pt >= th).astype(int), pt)
            row = {'model': model, 'view': view, 'split': 'S1', 'seed': seed,
                   'seconds': time.time() - t0, **m}
            rows.append(row)
            scvd.append_results(root, 'stage2_baselines', [row])   # checkpoint per run
            log.info('%-8s F_%-4s seed%d macro_f1=%.4f (%.0fs)', model, view, seed,
                     m['macro_f1'], row['seconds'])

2026-09-24 11:43:29,925 INFO RF-BR    F_pre  seed0 macro_f1=0.8075 (127s)
2026-09-24 11:45:36,735 INFO RF-BR    F_pre  seed1 macro_f1=0.8159 (127s)
2026-09-24 11:47:42,281 INFO RF-BR    F_pre  seed2 macro_f1=0.8173 (126s)
2026-09-24 11:48:16,024 INFO LGBM-BR  F_pre  seed0 macro_f1=0.8302 (34s)
2026-09-24 11:48:45,487 INFO LGBM-BR  F_pre  seed1 macro_f1=0.8302 (29s)
2026-09-24 11:49:15,166 INFO LGBM-BR  F_pre  seed2 macro_f1=0.8302 (30s)
2026-09-24 11:50:58,750 INFO CC-ens   F_pre  seed0 macro_f1=0.7205 (104s)
2026-09-24 11:52:43,702 INFO CC-ens   F_pre  seed1 macro_f1=0.7204 (105s)
2026-09-24 11:54:28,315 INFO CC-ens   F_pre  seed2 macro_f1=0.7221 (105s)
2026-09-24 11:58:57,927 INFO RAkEL    F_pre  seed0 macro_f1=0.6912 (270s)


In [ ]:
summ, sp = scvd.write_summary(root, 'stage2_baselines')
scvd.plot_model_macro_f1(summ, root, 's2_classical_macro_f1', log)
scvd.plot_perlabel_f1(summ, ['RF-BR', 'LGBM-BR', 'CC-ens'], root, 's2_classical_perlabel', log)
scvd.tex_table(summ, f'{root}/tables/stage2_baselines.tex',
               ['macro_f1', 'micro_f1', 'macro_ap', 'hamming'],
               'Classical and transformation baselines on S1 (mean over seeds).',
               'tab:stage2baselines')
summ.round(4)

In [ ]:
def push_dataset(root, stage, slug, title, log=None):
    """
    Push (or create) a Kaggle dataset from `root`.
    - If dataset already exists -> creates a new version.
    - If it doesn't exist yet -> initializes metadata and creates it.
    Returns the dataset URL, or None if no credentials/API available.
    """
    import os, json
    from kaggle.api.kaggle_api_extended import KaggleApi

    try:
        api = KaggleApi()
        api.authenticate()
    except Exception as e:
        if log: log.info("no kaggle credentials available (%s); skipping push", e)
        return None

    # Kaggle hard limits: slug 6-50 chars, title 6-50 chars
    if not (6 <= len(slug) <= 50):
        raise ValueError(f"slug must be 6-50 chars, got {len(slug)}: {slug!r}")
    if len(title) > 50:
        title = title[:50].rstrip()
        if log: log.info("title truncated to 50 chars: %r", title)
    if len(title) < 6:
        raise ValueError(f"title must be at least 6 chars, got {len(title)}: {title!r}")

    username = api.get_config_value('username')
    dataset_id = f"{username}/{slug}"
    url = f"https://www.kaggle.com/datasets/{dataset_id}"

    meta_path = os.path.join(root, 'dataset-metadata.json')

    if not os.path.exists(meta_path):
        api.dataset_initialize(root)

    with open(meta_path, 'r') as f:
        meta = json.load(f)
    meta['id'] = dataset_id
    meta['title'] = title
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)

    try:
        api.dataset_create_version(
            root,
            f"update: {stage} run",
            quiet=False,
            dir_mode='zip'
        )
        if log: log.info("pushed new version -> %s", url)
    except Exception as e:
        if log: log.info("create_version failed (%s); creating new dataset instead", e)
        api.dataset_create_new(
            folder=root,
            quiet=False,
            dir_mode='zip',
            public=False
        )
        if log: log.info("created dataset -> %s", url)

    return url

In [ ]:
title = 'SCVD Stage 2: classical baselines on DIVE'  # 42 chars, under the 50 cap
url = scvd.push_dataset(root, 'stage2_baselines', 'scvd-stage2-baselines', title, log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)